## Avaliando o agente

In [ ]:
from langsmith import traceable

In [ ]:
"""@traceable
async def agent_avaliado(question):
    answer = await agent.ainvoke(
    {
        "messages": 
            [HumanMessage(role="user",
                          content=question)],
        "todos": [],
    }
)
    return answer['messages'][-1].content"""


In [ ]:
# We'll first define a custom code evaluator, which are useful to measure deterministic or close-ended metrics.
def conciseness(outputs: dict) -> bool:
    words = outputs["output"].split(" ")
    return len(words) <= 200

LLM-as-a-Judge Evaluator
For open-ended metrics, it's can be powerful to use an LLM to score the outputs.

Let's use an LLM to check whether our application produces correct outputs. First, let's define a scoring schema for our LLM to adhere to in its response.

In [ ]:
from pydantic import BaseModel, Field

# Define a scoring schema that our LLM must adhere to
class CorrectnessScore(BaseModel):
    """Correctness score of the answer when compared to the reference answer."""
    score: int = Field(description="The score of the correctness of the answer, from 0 to 1")

In [ ]:
#from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import asyncio

async def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    prompt = """
    You are an expert data labeler evaluating model outputs for correctness. Your task is to assign a score based on the following rubric:

    <Rubric>
        A correct answer:
        - Provides accurate information
        - Uses suitable analogies and examples
        - Contains no factual errors
        - Is logically consistent

        When scoring, you should penalize:
        - Factual errors
        - Incoherent analogies and examples
        - Logical inconsistencies
    </Rubric>

    <Instructions>
        - Carefully read the input and output
        - Use the reference output to determine if the model output contains errors
        - Focus whether the model output uses accurate analogies and is logically consistent
    </Instructions>

    <Reminder>
        The analogies in the output do not need to match the reference output exactly. Focus on logical consistency.
    </Reminder>

    <input>
        {}
    </input>

    <output>
        {}
    </output>

    Use the reference outputs below to help you evaluate the correctness of the response:
    <reference_outputs>
        {}
    </reference_outputs>
    """.format(inputs["aswer_code"], outputs["output"], reference_outputs["response_code"])

    router_structured = LlmRouter(
            prompt,
            CorrectnessScore
        )
    response = await router_structured.llm_router()
    
    if isinstance(response, dict):
        response = response["score"]
    else:
        response = response.score
    return response

In [ ]:
import nest_asyncio

# Apply nest_asyncio at the start of your notebook
nest_asyncio.apply()
def correctness_sync(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    return asyncio.run(correctness(inputs, outputs, reference_outputs))

In [ ]:
response = correctness_sync({"aswer_code": "def add(a, b):\n    return a + b"}, {"output": "def add(a, b):\n    return a + b"}, {"response_code": "def add(a, b):\n    return a + b"})

INFO:__main__:Iniciando roteamento LLM
INFO:__main__:Tentando modelo Groq: moonshotai/kimi-k2-instruct-0905
INFO:__main__:Tentando modelo Groq: moonshotai/kimi-k2-instruct
INFO:__main__:Tentando modelo Groq: meta-llama/llama-4-scout-17b-16e-instruct
INFO:__main__:Tentando modelo Groq: openai/gpt-oss-120b
INFO:__main__:Tentando modelo HuggingFace: Qwen/Qwen3-235B-A22B-Instruct-2507
INFO:__main__:Sucesso com modelo HuggingFace: Qwen/Qwen3-235B-A22B-Instruct-2507


In [ ]:
"""async def run(inputs: dict):
    return await agent_avaliado(inputs["aswer_code"])"""


async def agent_teste(inputs: dict):
    
    return "Ola mundo"

async def run(inputs: dict):
    return await agent_teste(inputs["aswer_code"])


In [ ]:
from langsmith import evaluate, aevaluate
results = asyncio.run(aevaluate(
    run,
    data=dataset_name_10,
    evaluators=[correctness],
    experiment_prefix="test-human-eval-10",
))

View the evaluation results for experiment: 'test-human-eval-10-e25def82' at:
https://smith.langchain.com/o/d9ab5018-45a8-5efd-961d-320c87839c87/datasets/0b278319-9299-4f5c-8ac3-d068c769469f/compare?selectedSessions=63ebe204-9c56-47fc-b80b-d68588351b6c




0it [00:00, ?it/s]INFO:__main__:Iniciando roteamento LLM
INFO:__main__:Tentando modelo Groq: moonshotai/kimi-k2-instruct-0905
INFO:__main__:Tentando modelo Groq: moonshotai/kimi-k2-instruct
INFO:__main__:Tentando modelo Groq: meta-llama/llama-4-scout-17b-16e-instruct
INFO:__main__:Tentando modelo Groq: openai/gpt-oss-120b
INFO:__main__:Tentando modelo HuggingFace: Qwen/Qwen3-235B-A22B-Instruct-2507
INFO:__main__:Sucesso com modelo HuggingFace: Qwen/Qwen3-235B-A22B-Instruct-2507
1it [00:01,  1.99s/it]INFO:__main__:Iniciando roteamento LLM
INFO:__main__:Tentando modelo Groq: moonshotai/kimi-k2-instruct-0905
INFO:__main__:Tentando modelo Groq: moonshotai/kimi-k2-instruct
INFO:__main__:Tentando modelo Groq: meta-llama/llama-4-scout-17b-16e-instruct
INFO:__main__:Tentando modelo Groq: openai/gpt-oss-120b
INFO:__main__:Tentando modelo HuggingFace: Qwen/Qwen3-235B-A22B-Instruct-2507
INFO:__main__:Sucesso com modelo HuggingFace: Qwen/Qwen3-235B-A22B-Instruct-2507
2it [00:02,  1.21s/it]INFO:__m

In [ ]:
# TODO: Copy your experiment name here
experiments_names = ["openai/gpt-oss-120b-human-eval-code-0f7df7f4"]

# Set this to load expt results
for experiment_name in experiments_names:
    print("Loading results for experiment:", experiment_name)
    experiment_results = client.read_project(project_name=experiment_name, include_stats=True)
    print("Latency p50:", experiment_results.latency_p50)
    print("Latency p99:", experiment_results.latency_p99)
    print("Token Usage:", experiment_results.total_tokens)
    print("Feedback Stats:", experiment_results.feedback_stats)
    print("*" * 50)

Loading results for experiment: openai/gpt-oss-120b-human-eval-code-0f7df7f4
Latency p50: 0:02:00.906000
Latency p99: 0:05:09.829270
Token Usage: 237234
Feedback Stats: {'correctness': {'n': 10, 'avg': 0.8, 'stdev': 0.39999999999999997, 'errors': 0, 'values': {}, 'type': 'primary'}}
**************************************************


In [ ]:
experiment_results.dict()

{'id': UUID('0c58ecc6-7960-46ef-bdb9-44191066c11f'),
 'start_time': datetime.datetime(2025, 10, 2, 12, 47, 36, 944679, tzinfo=datetime.timezone.utc),
 'end_time': None,
 'description': None,
 'name': 'openai/gpt-oss-120b-human-eval-code-0f7df7f4',
 'extra': {'metadata': {'git': {'tags': None,
    'dirty': True,
    'branch': 'secundario',
    'commit': '47b18b6248c690a39e12ddf4182201433d49be46',
    'repo_name': 'AgenteCodificaoLangGraph',
    'remote_url': 'https://github.com/Jeferson100/Code-Agent.git',
    'author_name': 'JEFERSON DIONEI SEHNEM',
    'commit_time': '1759163044',
    'author_email': 'sehnemjeferson@gmail.com'},
   'revision_id': '47b18b6-dirty',
   'dataset_splits': ['base'],
   'dataset_version': '2025-10-01T17:46:07.806191+00:00',
   'num_repetitions': 1}},
 'tenant_id': UUID('d9ab5018-45a8-5efd-961d-320c87839c87'),
 'reference_dataset_id': UUID('0b278319-9299-4f5c-8ac3-d068c769469f'),
 'run_count': 10,
 'latency_p50': datetime.timedelta(seconds=120, microseconds=9

### Acessando datasets do LangSmith

In [ ]:
# List all datasets
from langsmith import Client

client = Client()

datasets = client.list_datasets()
for dataset in datasets:
    print(dataset.id, dataset.name)

0b278319-9299-4f5c-8ac3-d068c769469f Human-Eval-Code-10
b9dd1aaa-017b-4280-96ca-cd6f094773fa deep_research_supervisor_parallelism
5873a0fe-22fd-4a3d-8eb6-86330c26e52f deep_research_agent_termination
2bdcce58-182b-4a36-98a5-74720bc26a05 deep_research_scoping
27916b23-40e0-4874-8d5c-ddbebfcb8958 E-mail Triage Evaluation
b746e978-0744-44de-9f42-6cbd42bbd1e3 agents-from-scratch.test_response
623cff70-db0b-4993-bfe4-6ee88d1a46ac agents-from-scratch.test_tools
a2defcc0-e281-43c6-b351-c7930a9307ac Financial Advisory RAG Evaluation
77fa7a2a-4ece-4096-96fa-ed9d1d9cff81 Healthcare Agent Trajectory Evaluation
43be1111-6a51-4928-a33e-940a33a8b40a Reasoning and Bias
ba1609ff-70a6-4a64-b849-1892bbd24274 QA Example Dataset
aea62e89-af3d-4bb0-98e6-275fd643bfe5 Sample dataset
1bf6cc9b-eae8-404b-adca-0083f19e6e39 Human-Eval-Code
da848d07-6fa9-4ffe-a503-570e10840ac0 Insurance Claims


In [ ]:
# List one dataset by ID
dataset = client.read_dataset(dataset_id="0b278319-9299-4f5c-8ac3-d068c769469f")
print(dataset)


name='Human-Eval-Code-10' description='The HumanEval dataset released by OpenAI includes 164 programming problems with a function sig- nature, docstring, body, and several unit tests. They were handwritten to ensure not to be included in the training set of code generation models.' data_type=<DataType.kv: 'kv'> id=UUID('0b278319-9299-4f5c-8ac3-d068c769469f') created_at=datetime.datetime(2025, 10, 1, 17, 46, 7, 146797, tzinfo=datetime.timezone.utc) modified_at=datetime.datetime(2025, 10, 1, 17, 46, 7, 146797, tzinfo=datetime.timezone.utc) example_count=10 session_count=1 last_session_start_time=datetime.datetime(2025, 10, 2, 12, 47, 36, 944679) inputs_schema=None outputs_schema=None transformations=None


In [ ]:
examples = client.list_examples(dataset_id="0b278319-9299-4f5c-8ac3-d068c769469f")
for example in examples:
    print(example.inputs, example.outputs)

{'aswer_code': 'from typing import List\n\n\ndef below_zero(operations: List[int]) -> bool:\n    """ You\'re given a list of deposit and withdrawal operations on a bank account that starts with\n    zero balance. Your task is to detect if at any point the balance of account fallls below zero, and\n    at that point function should return True. Otherwise it should return False.\n    >>> below_zero([1, 2, 3])\n    False\n    >>> below_zero([1, 2, -4, 5])\n    True\n    """\n'} {'response_code': "\n\nMETADATA = {\n    'author': 'jt',\n    'dataset': 'test'\n}\n\n\ndef check(candidate):\n    assert candidate([]) == False\n    assert candidate([1, 2, -3, 1, 2, -3]) == False\n    assert candidate([1, 2, -4, 5, 6]) == True\n    assert candidate([1, -1, 2, -2, 5, -5, 4, -4]) == False\n    assert candidate([1, -1, 2, -2, 5, -5, 4, -5]) == True\n    assert candidate([1, -2, 2, -2, 5, -5, 4, -4]) == True\n"}
{'aswer_code': 'from typing import List\n\n\ndef separate_paren_groups(paren_string: str)

In [ ]:
experiment_results.dict()

{'id': UUID('0c58ecc6-7960-46ef-bdb9-44191066c11f'),
 'start_time': datetime.datetime(2025, 10, 2, 12, 47, 36, 944679, tzinfo=datetime.timezone.utc),
 'end_time': None,
 'description': None,
 'name': 'openai/gpt-oss-120b-human-eval-code-0f7df7f4',
 'extra': {'metadata': {'git': {'tags': None,
    'dirty': True,
    'branch': 'secundario',
    'commit': '47b18b6248c690a39e12ddf4182201433d49be46',
    'repo_name': 'AgenteCodificaoLangGraph',
    'remote_url': 'https://github.com/Jeferson100/Code-Agent.git',
    'author_name': 'JEFERSON DIONEI SEHNEM',
    'commit_time': '1759163044',
    'author_email': 'sehnemjeferson@gmail.com'},
   'revision_id': '47b18b6-dirty',
   'dataset_splits': ['base'],
   'dataset_version': '2025-10-01T17:46:07.806191+00:00',
   'num_repetitions': 1}},
 'tenant_id': UUID('d9ab5018-45a8-5efd-961d-320c87839c87'),
 'reference_dataset_id': UUID('0b278319-9299-4f5c-8ac3-d068c769469f'),
 'run_count': 10,
 'latency_p50': datetime.timedelta(seconds=120, microseconds=9